In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Paths -- edit these for your environment.
RAW_DIR  = "/PATH/TO/raw"              # raw MGB extracts (edw/, clarity/, Datamart/)
DATA_DIR = "/PATH/TO/data"             # project data root; lookup tables live here
WORK_DIR = f"{DATA_DIR}/20250604"      # identified intermediate outputs -- keep on the secure server
DEID_DIR = f"{WORK_DIR}/deidentified"  # de-identified outputs and ID / date-shift maps

In [ ]:
base_cohort = pd.read_csv(f'{WORK_DIR}/microbiology_cultures_cohort.csv')
base_cohort = base_cohort[['anon_id',
                           'pat_enc_csn_id_coded',
                           'order_proc_id_coded',
                           'order_time_jittered_utc',]].copy()

CCSR_code = pd.read_csv(f'{DATA_DIR}/ahrq_ccsr_diagnosis.csv')
CCSR_code = CCSR_code[['ICD10','Category_Description']].copy()

elixhauser_codes = pd.read_csv(f'{DATA_DIR}/elixhauser-comorbidity-components.csv')
elixhauser_codes['ICD10'] = elixhauser_codes['icd10']
elixhauser_codes = elixhauser_codes[['ICD10','category']].copy()

diagnosis = pd.read_csv(f'{RAW_DIR}/edw/diagnoses/MGB_Dx_20241105.csv')
diagnosis = diagnosis.rename(columns = {'PatientID':'anon_id',
                                        'PatientEncounterID':'pat_enc_csn_id_coded',
                                        'icd10':'ICD10',})
diagnosis = diagnosis[['anon_id',
                       'pat_enc_csn_id_coded',
                       'ICD10']].copy()


In [ ]:
AHRQ_CCSR = diagnosis.merge(CCSR_code,
                            on = ['ICD10'],
                            how = 'inner')
AHRQ_CCSR = AHRQ_CCSR.rename(columns={'Category_Description':'category'})

elixhauser_components = diagnosis.merge(elixhauser_codes,
                                        on = ['ICD10'],
                                        how = 'inner')

all_components = pd.concat([elixhauser_components, AHRQ_CCSR], 
                           ignore_index=True)
all_components

In [ ]:
microbiology_comorbidity_augmented = base_cohort.merge(all_components, 
                                                       on=['anon_id','pat_enc_csn_id_coded'], 
                                                       how='left')

microbiology_comorbidity_augmented

In [ ]:
microbiology_comorbidity_augmented.to_csv(f'{WORK_DIR}/microbiology_cultures_comorbidity.csv',
                                          index = False)

## Deidentification

In [ ]:
# Load the saved mappings (assume you saved as CSVs before)
anon_map_df = pd.read_csv(f'{DEID_DIR}/anon_id_map.csv')
enc_map_df = pd.read_csv(f'{DEID_DIR}/enc_id_map.csv')
order_proc_map_df = pd.read_csv(f'{DEID_DIR}/order_proc_id_map.csv')

In [ ]:
# Convert to Dict
anon_id_map = dict(zip(anon_map_df['anon_id'], anon_map_df['anon_id_deid']))
enc_id_map = dict(zip(enc_map_df['pat_enc_csn_id_coded'], enc_map_df['pat_enc_csn_id_deid']))
order_proc_map = dict(zip(order_proc_map_df['order_proc_id_coded'], order_proc_map_df['order_proc_id_deid']))

# Apply the mappings
final_table_deid = microbiology_comorbidity_augmented.copy()
final_table_deid['anon_id'] = final_table_deid['anon_id'].map(anon_id_map)
final_table_deid['pat_enc_csn_id_coded'] = final_table_deid['pat_enc_csn_id_coded'].map(enc_id_map)
final_table_deid['order_proc_id_coded'] = final_table_deid['order_proc_id_coded'].map(order_proc_map)

final_table_deid

In [ ]:
final_table_deid.to_csv(f'{DEID_DIR}/microbiology_cultures_comorbidity.csv',
                        index = False)